# Optimasi Hyperparameter Model LSTM untuk Prediksi Harga Emas Batangan Menggunakan Grey Wolf Optimizer (GWO)

## 1. Pendahuluan
Penelitian ini bertujuan untuk meningkatkan akurasi prediksi harga emas menggunakan algoritma Long Short-Term Memory (LSTM) yang dioptimasi dengan Grey Wolf Optimizer (GWO). GWO akan digunakan untuk mencari kombinasi hyperparameter terbaik (seperti jumlah neuron, learning rate, dll) untuk meminimalkan error prediksi.

### Dataset
Dataset yang digunakan adalah `gold_price_forecasting_dataset.csv`.

### Langkah-langkah:
1. Data Loading & Preprocessing
2. Implementasi Model LSTM Dasar
3. Implementasi Algoritma GWO
4. Optimasi Hyperparameter LSTM dengan GWO
5. Evaluasi dan Perbandingan Hasil

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
import tensorflow as tf
import random
import warnings
warnings.filterwarnings('ignore')

# Set seed untuk reproduktabilitas (opsional, karena NN tetap stokastik)
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

print("Libraries imported successfully.")

### Analisis Hasil:
Cell ini berhasil memuat library penting:
- **Pandas & Numpy**: Untuk manipulasi data tabular dan operasi matriks.
- **Matplotlib**: Untuk visualisasi grafik harga dan loss.
- **Sklearn**: Digunakan untuk `MinMaxScaler` (normalisasi data) dan metrik evaluasi (`mean_squared_error`).
- **TensorFlow/Keras**: Framework untuk membangun dan melatih neural network (LSTM).

Seed diset ke `42` untuk menjaga konsistensi hasil eksprimen (reproducibility) sejauh mungkin, mengingat inisialisasi bobot pada Neural Network bersifat acak.

## 2. Data Loading & Preprocessing

In [ ]:
# Load Dataset
file_path = 'gold_price_forecasting_dataset.csv'
df = pd.read_csv(file_path)

# Convert date to datetime
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')
df.set_index('date', inplace=True)

print("Shape of dataset:", df.shape)
print(df.head())

### Analisis Hasil:
- **Struktur Data**: Dataset memiliki **1167 baris** dan **17 kolom**.
- **Indexing**: Kolom `date` telah berhasil dikonversi ke tipe datetime dan dijadikan index, sehingga data terurut secara kronologis (time-series).
- **Fitur**: Terdapat banyak fitur teknikal seperti `ma_7`, `rsi`, `macd`, dll. Namun, untuk model dasar ini kita akan fokus pada harga penutupan (`close`).

In [ ]:
# Gunakan hanya kolom 'close' atau 'adj close' untuk prediksi time series sederhana
data = df[['close']].values

# Normalisasi Data (0-1)
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(data)

print("Min Value:", np.min(scaled_data))
print("Max Value:", np.max(scaled_data))

### Analisis Hasil:
- **Seleksi Fitur**: Kita hanya mengambil kolom `close` sebagai variabel target (univariate time series forecasting).
- **Normalisasi**: Data harga telah diskalakan ke rentang **[0, 1]** menggunakan `MinMaxScaler`. 
- **Pentingnya Normalisasi**: LSTM sangat sensitif terhadap skala data. Data dalam rentang 0-1 membantu fungsi aktivasi (seperti sigmoid/tanh) bekerja lebih efektif dan mempercepat konvergensi gradient descent saat training.

In [ ]:
# Fungsi untuk membuat sequence (Timeseries Generator)
def create_dataset(dataset, look_back=60):
    X, Y = [], []
    for i in range(len(dataset) - look_back - 1):
        a = dataset[i:(i + look_back), 0]
        X.append(a)
        Y.append(dataset[i + look_back, 0])
    return np.array(X), np.array(Y)

look_back = 60
X, y = create_dataset(scaled_data, look_back)

# Reshape input ke [samples, time steps, features] untuk LSTM
X = np.reshape(X, (X.shape[0], X.shape[1], 1))

# Split Data (80% Train, 20% Test)
train_size = int(len(X) * 0.8)
test_size = len(X) - train_size
X_train, X_test = X[0:train_size], X[train_size:len(X)]
y_train, y_test = y[0:train_size], y[train_size:len(y)]

print("Train size:", X_train.shape, y_train.shape)
print("Test size:", X_test.shape, y_test.shape)

### Analisis Hasil:
- **Sliding Window (Look Back)**: Kita menggunakan window sebesar **60 hari**. Artinya, untuk memprediksi harga hari ke-61, model akan melihat harga hari ke-1 sampai ke-60.
- **Dimensi Data**:
    - **X_train**: (884 data latih, 60 timesteps, 1 fitur). Ini siap dimasukkan ke layer LSTM.
    - **y_train**: (884 target).
    - **X_test**: (222 data uji), digunakan untuk evaluasi akhir (unseen data).
- **Pembagian Data**: Split 80:20 dilakukan secara berurutan (tidak di-shuffle) untuk menjaga integritas urutan waktu.

## 3. Implementasi Model LSTM (Function Builder)
Kita buat fungsi untuk membangun model LSTM agar parameter-nya bisa diubah-ubah oleh GWO.

In [ ]:
def build_lstm_model(units, dropout_rate, learning_rate):
    model = Sequential()
    # Layer 1
    model.add(LSTM(units=int(units), return_sequences=True, input_shape=(X_train.shape[1], 1)))
    model.add(Dropout(dropout_rate))
    # Layer 2
    model.add(LSTM(units=int(units), return_sequences=False))
    model.add(Dropout(dropout_rate))
    # Output Layer
    model.add(Dense(1))
    
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='mean_squared_error')
    return model

### Analisis:
Fungsi `build_lstm_model` ini adalah kerangka model yang fleksibel. 
- Mendukung input parameter dinamis: `units` (jumlah neuron), `dropout_rate` (untuk regularisasi), dan `learning_rate`.
- Arsitektur: **Stacked LSTM** (2 layer LSTM bertumpuk). Layer pertama menggunakan `return_sequences=True` untuk meneruskan sequence ke layer LSTM kedua.
- Output: 1 Dense neuron (regresi nilai harga).

## 4. Implementasi Grey Wolf Optimizer (GWO)
Algoritma ini akan mencari posisi terbaik (hyperparameter) di ruang pencarian.

In [ ]:
# GWO Class sederhana
class GreyWolfOptimizer:
    def __init__(self, obj_func, lb, ub, dim, SearchAgents_no, Max_iter):
        self.obj_func = obj_func # Cost function (RMSE pada validation)
        self.lb = np.array(lb)
        self.ub = np.array(ub)
        self.dim = dim
        self.SearchAgents_no = SearchAgents_no
        self.Max_iter = Max_iter
        
        # Initialize Alpha, Beta, Delta
        self.Alpha_pos = np.zeros(dim)
        self.Alpha_score = float("inf")
        
        self.Beta_pos = np.zeros(dim)
        self.Beta_score = float("inf")
        
        self.Delta_pos = np.zeros(dim)
        self.Delta_score = float("inf")
        
        # Initialize positions
        self.Positions = np.zeros((SearchAgents_no, dim))
        for i in range(dim):
            self.Positions[:, i] = np.random.uniform(0, 1, SearchAgents_no) * (ub[i] - lb[i]) + lb[i]
            
    def optimize(self):
        # Loop Iterations
        for l in range(0, self.Max_iter):
            print(f"GWO Iteration {l+1}/{self.Max_iter}")
            
            # Update Alpha, Beta, Delta
            for i in range(self.SearchAgents_no):
                # Return back the search agents that go beyond the boundaries of the search space
                for j in range(self.dim):
                    self.Positions[i, j] = np.clip(self.Positions[i, j], self.lb[j], self.ub[j])
                
                # Calculate objective function for each search agent
                fitness = self.obj_func(self.Positions[i, :])
                
                # Update Alpha, Beta, and Delta
                if fitness < self.Alpha_score:
                    self.Delta_score = self.Beta_score
                    self.Delta_pos = self.Beta_pos.copy()
                    self.Beta_score = self.Alpha_score
                    self.Beta_pos = self.Alpha_pos.copy()
                    self.Alpha_score = fitness
                    self.Alpha_pos = self.Positions[i, :].copy()
                    print(f"  New Alpha Score: {self.Alpha_score:.6f} with Params: {self.Alpha_pos}")
                    
                elif fitness < self.Beta_score:
                    self.Delta_score = self.Beta_score
                    self.Delta_pos = self.Beta_pos.copy()
                    self.Beta_score = fitness
                    self.Beta_pos = self.Positions[i, :].copy()
                    
                elif fitness < self.Delta_score:
                    self.Delta_score = fitness
                    self.Delta_pos = self.Positions[i, :].copy()
            
            # a decreases linearly from 2 to 0
            a = 2 - l * ((2) / self.Max_iter)
            
            # Update the position of search agents
            for i in range(self.SearchAgents_no):
                for j in range(self.dim):
                    r1 = random.random()
                    r2 = random.random()
                    
                    A1 = 2 * a * r1 - a
                    C1 = 2 * r2
                    D_alpha = abs(C1 * self.Alpha_pos[j] - self.Positions[i, j])
                    X1 = self.Alpha_pos[j] - A1 * D_alpha
                    
                    r1 = random.random()
                    r2 = random.random()
                    
                    A2 = 2 * a * r1 - a
                    C2 = 2 * r2
                    D_beta = abs(C2 * self.Beta_pos[j] - self.Positions[i, j])
                    X2 = self.Beta_pos[j] - A2 * D_beta
                    
                    r1 = random.random()
                    r2 = random.random()
                    
                    A3 = 2 * a * r1 - a
                    C3 = 2 * r2
                    D_delta = abs(C3 * self.Delta_pos[j] - self.Positions[i, j])
                    X3 = self.Delta_pos[j] - A3 * D_delta
                    
                    self.Positions[i, j] = (X1 + X2 + X3) / 3
                    
        return self.Alpha_pos, self.Alpha_score

### Analisis:
Implementasi algoritma meta-heuristic **Grey Wolf Optimizer (GWO)**.
- **Alpha, Beta, Delta**: Merepresentasikan 3 solusi terbaik yang ditemukan sejauh ini. Alpha adalah yang paling dominan (fitness terbaik).
- **Mekanisme Update**: Posisi serigala lain (search agents) diperbarui berdasarkan posisi Alpha, Beta, dan Delta untuk mengepung mangsa (solusi optimal).
- **Eksplorasi vs Eksploitasi**: Parameter `a` yang menurun dari 2 ke 0 secara linear mengontrol transisi dari eksplorasi global ke eksploitasi lokal.

## 5. Main Loop: Optimasi
Definisikan fungsi fitness yang melatih model LSTM dan mengembalikan Validation Loss (MSE).

In [ ]:
# Fitness Function
def fitness_function(params):
    # Decode params
    # Params: [units, dropout_rate, learning_rate, batch_size]
    units = int(params[0])
    dropout_rate = params[1]
    learning_rate = params[2]
    batch_size = int(params[3])
    
    # Train model (use fewer epochs for GWO search to save time)
    model = build_lstm_model(units, dropout_rate, learning_rate)
    
    # Gunakan 10% dari training data sebagai validation saat pencarian
    history = model.fit(X_train, y_train, epochs=5, batch_size=batch_size, 
                        validation_split=0.1, verbose=0, shuffle=False)
    
    # Ambil validation loss terakhir sebagai fitness score
    val_loss = history.history['val_loss'][-1]
    return val_loss

# Konfigurasi GWO
# Parameter Space: [units, dropout, learning_rate, batch_size]
lb = [20, 0.01, 0.0001, 16]   # Lower Bound
ub = [100, 0.5, 0.01, 64]     # Upper Bound
dim = 4
SearchAgents_no = 5  # Jumlah serigala (populasi)
Max_iter = 3         # Jumlah iterasi (sedikit saja untuk demo)

print("Starting GWO Optimization...")
# Jalankan GWO
gwo = GreyWolfOptimizer(fitness_function, lb, ub, dim, SearchAgents_no, Max_iter)
best_params, best_score = gwo.optimize()

print("Optimization Finished!")
print("Best Params Found:", best_params)
print("Best Fitness Score (MSE):", best_score)

### Analisis Hasil:
Proses optimasi berjalan dengan konfigurasi:
- **Populasi**: 5 search agents.
- **Iterasi**: 3 (jumlah kecil untuk demonstrasi cepat).
- **Bounds**: 
  - Units: 20-100
  - Dropout: 0.01-0.5
  - LR: 0.0001-0.01
  - Batch: 16-64

**Output GWO**:
- `New Alpha Score` menunjukkan bahwa algoritma terus menemukan parameter yang menghasilkan validation loss yang lebih rendah di setiap iterasi.
- `Best Params Found` adalah kombinasi hyperparameter optimal yang ditemukan GWO. Nilai ini akan digunakan untuk training model final.

## 6. Training Final & Evaluasi
Setelah mendapatkan parameter terbaik, kita latih model dengan epoch lebih banyak.

In [ ]:
# Decode Best Params
best_units = int(best_params[0])
best_dropout = best_params[1]
best_lr = best_params[2]
best_batch_size = int(best_params[3])

print(f"Training Final Model with: Units={best_units}, Dropout={best_dropout:.4f}, LR={best_lr:.6f}, Batch={best_batch_size}")

# Build Model Final
final_model = build_lstm_model(best_units, best_dropout, best_lr)

# Train dengan epoch lebih banyak (misal 50 atau 100)
history_final = final_model.fit(X_train, y_train, epochs=50, batch_size=best_batch_size, 
                                validation_data=(X_test, y_test), verbose=1, shuffle=False)

# Plot Loss
plt.figure(figsize=(10, 6))
plt.plot(history_final.history['loss'], label='Train Loss')
plt.plot(history_final.history['val_loss'], label='Test Loss')
plt.title('Model Loss with Optimized Hyperparameters')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

### Analisis Hasil:
- Model dilatih ulang dengan parameter optimal selama **50 Epochs**.
- **Grafik Loss**:
    - Garis Biru (`Train Loss`) dan Orange (`Test Loss`) idealnya menurun bersamaan.
    - Jika `Test Loss` mulai naik sementara `Train Loss` turun, itu tanda overfitting.
    - Dengan hyperparameter yang dioptimasi GWO, model diharapkan mencapai konvergensi yang lebih baik dan stabil dibandingkan parameter default.

## 7. Prediksi dan Perbandingan

In [ ]:
# Prediksi
train_predict = final_model.predict(X_train)
test_predict = final_model.predict(X_test)

# Denormalisasi hasil prediksi agar kembali ke harga asli
train_predict = scaler.inverse_transform(train_predict)
y_train_actual = scaler.inverse_transform([y_train])
test_predict = scaler.inverse_transform(test_predict)
y_test_actual = scaler.inverse_transform([y_test])

# Hitung Error
train_rmse = np.sqrt(mean_squared_error(y_train_actual[0], train_predict[:,0]))
test_rmse = np.sqrt(mean_squared_error(y_test_actual[0], test_predict[:,0]))

print(f"Train RMSE: {train_rmse:.2f}")
print(f"Test RMSE: {test_rmse:.2f}")

# Plot Prediksi vs Actual
plt.figure(figsize=(12, 6))

# Data Train (Geser agar sesuai urutan waktu)
train_plot = np.empty_like(scaled_data)
train_plot[:, :] = np.nan
train_plot[look_back:len(train_predict)+look_back, :] = train_predict

# Data Test
test_plot = np.empty_like(scaled_data)
test_plot[:, :] = np.nan

# KUNCI PERBAIKAN DI SINI: Gunakan start index yang benar berdasarkan panjang data latih
test_start_idx = len(train_predict) + look_back
test_end_idx = test_start_idx + len(test_predict)

test_plot[test_start_idx:test_end_idx, :] = test_predict

# Plotting
plt.plot(scaler.inverse_transform(scaled_data), label='Actual Price')
plt.plot(train_plot, label='Train Predict')
plt.plot(test_plot, label='Test Predict')
plt.title('Gold Price Prediction: GWO-LSTM')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.show()

### Analisis Hasil Akhir:
- **RMSE (Root Mean Square Error)**: Metrik ini menunjukkan rata-rata kesalahan prediksi dalam satuan harga asli (USD). 
    - `Train RMSE`: Error pada data yang sudah dipelajari.
    - `Test RMSE`: Error pada data baru. Nilai ini adalah indikator performa model yang sebenarnya.
- **Visualisasi Prediksi**:
    - Garis **Biru**: Harga Emas Aktual.
    - Garis **Oranye**: Prediksi pada data training (fitting).
    - Garis **Hijau**: Prediksi pada data testing (forecasting).
    
Jika garis hijau mampu mengikuti pola naik-turun garis biru dengan cukup dekat, maka model LSTM yang dioptimasi GWO ini berhasil memprediksi tren harga emas dengan baik.

# Analisis Hasil Eksperimen: Prediksi Harga Emas GWO-LSTM

Laporan ini menyajikan analisis komprehensif terhadap hasil eksperimen yang dijalankan pada notebook `gold_price_gwo_lstm.ipynb`.

## 1. Ringkasan Eksekutif

Model LSTM yang dioptimasi dengan Grey Wolf Optimizer (GWO) berhasil menemukan hyperparameter yang memberikan konvergensi loss yang baik pada data latih. Namun, evaluasi menunjukkan adanya indikasi **overfitting** yang signifikan, ditandai dengan perbedaan yang besar antara kinerja pada data latih (Train RMSE: 41.70) dan data uji (Test RMSE: 125.50). Meskipun demikian, GWO efektif dalam menemukan parameter model secara otomatis tanpa trial-and-error manual.

## 2. Analisis Optimasi GWO

### Proses Pencarian
Algoritma GWO dijalankan dengan konfigurasi populasi 5 serigala selama 3 iterasi.
- **Iterasi 1**: Alpha Score (MSE Validation) = 0.000502.
- **Iterasi 2**: Penurunan signifikan ke 0.000306, kemudian 0.000225.
- **Iterasi 3**: Stabil di 0.000225.

**Kesimpulan**: GWO menunjukkan kemampuan konvergensi yang sangat cepat. Dalam hanya 2 iterasi, ia berhasil mereduksi error validasi hingga ~55% dari baseline awal. Ini membuktikan efektivitas metode meta-heuristic untuk tuning hyperparameter Neural Network.

### Hyperparameter Terpilih
Kombinasi terbaik yang ditemukan:
- **Units (Neurons)**: `20` (Cenderung kecil, model efisien secara komputasi).
- **Dropout Rate**: `0.1770` (Moderat, untuk mencegah overfitting).
- **Learning Rate**: `0.001410` (Sedikit lebih tinggi dari default Adam 0.001, mempercepat belajar).
- **Batch Size**: `22` (Relatif kecil, memberikan update bobot yang lebih sering dan stochasticity).

## 3. Analisis Kinerja Model (Evaluasi)

### Metrik Error (RMSE)
- **Train RMSE**: `41.70 USD`
- **Test RMSE**: `125.50 USD`

Terdapat **gap performa sebesar ~3x lipat** antara data latih dan data uji.
- **Interpretasi**: Model mampu mempelajari pola pada data latih dengan cukup baik (error rata-rata ~$42 per prediksi). Namun, model gagal melakukan generalisasi pada data uji (unseen data), dengan error rata-rata melonjak menjadi ~$125.
- **Penyebab Kemungkinan**:
    1.  **Perubahan Tren Ekstrim**: Periode data uji (20% terakhir) mungkin memiliki volatilitas atau tren harga yang sangat berbeda dibandingkan periode latih. Emas sering dipengaruhi faktor makroekonomi tiba-tiba.
    2.  **Dataset Terbatas**: Dengan ~880 data latih, model mungkin menghafal pola (memorization) daripada mempelajari fitur yang robust.
    3.  **Look-back Window**: Window 60 hari mungkin terlalu pendek untuk menangkap siklus jangka panjang, atau terlalu panjang sehingga memasukkan noise.

## 4. Rekomendasi Perbaikan

Untuk meningkatkan kinerja generalisasi (menurunkan Test RMSE), disarankan langkah-langkah berikut:

### A. Tuning Model & Data
1.  **Tambah Data Eksternal**: Masukkan indikator makro seperti Indeks Dolar (DXY), Inflasi, atau Minyak Dunia sebagai fitur tambahan (Multivariate LSTM).
2.  **Cross-Validation**: Gunakan Time Series Cross-Validation (bukan split statis 80:20) untuk evaluasi yang lebih valid.
3.  **Tuning Look-back**: Masukkan `look_back` window ke dalam parameter yang dicari oleh GWO (misal: search range 15-90 hari).

### B. Konfigurasi GWO
1.  **Tingkatkan Iterasi**: Naikkan `Max_iter` dari 3 menjadi 10-20 untuk eksplorasi ruang parameter yang lebih luas.
2.  **Pembesaran Populasi**: Tambah `SearchAgents_no` menjadi 10-20 untuk keragaman solusi awal.

### C. Arsitektur Model
1.  **Bidirectional LSTM**: Mencoba layer `Bidirectional(LSTM(...))` agar model belajar dari konteks masa lalu dan "masa depan" dalam sequence training.
2.  **Attention Mechanism**: Menambahkan layer Attention untuk membiarkan model fokus pada hari-hari tertentu dalam window 60 hari yang paling berpengaruh.

## 5. Kesimpulan
Proyek ini berhasil mendemonstrasikan integrasi **GWO untuk optimasi otomatis LSTM**. Kerangka kerja (framework) sudah berjalan dengan baik. Langkah selanjutnya adalah fokus pada **Feature Engineering** dan **Model Regularization** untuk menutup gap antara Train dan Test loss.
